# C10-competition-craft — Practice p18

**Type:** challenge · **Difficulty:** advanced · **Budget:** 105 minutes

**Concepts:** writeup-quality, colab-markdown-solution-authoring, markdown-code-snippets, markdown-math-formulae, colab-coding-submission, cpu-and-gpu-round-boundary, train-test-split, class-imbalance, accuracy-precision-recall, f1-macro, knn, feature-scaling, sklearn-pipelines

Audit the six-cell raw Colab transcript, map every defect to a repair, and produce the corrected submission. Reproduce the raw odd-`k` search only as an audit; separately select over exactly `K_GRID = [5, 7, 9, 11, 15]` by true macro-F1 with the smallest-`k` tie rule, then refit on all 600 rows.

Write exact artifacts `p18_audit.csv` and `p18_predictions.csv`, preserve the fresh-run stage trace, repair the rendered code and math, and state the CPU/GPU boundary.

## Row 2 — Complete cell audit

`cell_audit` has one complete record per numbered source cell, and `cell_plan` describes the corrected mixed-cell order.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
K_GRID = np.array([5, 7, 9, 11, 15])
df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
stage_trace = ["setup"]

In [ ]:
cell_audit = [
    {"cell_number": 1, "submitted_type": "code", "required_type": "split",
     "defect": "writeup and unrendered formula are code comments; claims are vague and unsupported; formula symbols are undefined; RUNTIME enables an illegal Round 1 GPU",
     "repair": "move Approach and Intuition, rendered macro-F1, and policy to separate text cells; name the bounded recipe and measured score; define symbols; remove GPU code and state the exact boundary"},
    {"cell_number": 2, "submitted_type": "text", "required_type": "split",
     "defect": "the fenced excerpt communicates the function but does not create an executable definition or verify its contract",
     "repair": "keep the matched python fence in text and add an identical executable predict_labels definition plus contract checks in code"},
    {"cell_number": 3, "submitted_type": "code", "required_type": "code",
     "defect": "searches undeclared odd k values 1 through 51; ranks by accuracy via .score but labels it macro-F1; raw max favors larger k on a score tie; leaves the chosen model fit only on the training carve; omits bounded log and stage trace",
     "repair": "reproduce that path only for audit; separately evaluate exact K_GRID by macro-F1 with smallest-k ties; record selection_log; append the select stage; refit the corrected winner on all rows"},
    {"cell_number": 4, "submitted_type": "text", "required_type": "text",
     "defect": "uses a binary F1 expression without needed parentheses while claiming macro averaging; leaves symbols undefined; malformed one-line python fence does not render reliably",
     "repair": "render the macro average in display math, define K and per-class F1, interpret equal weighting, and use matched multiline python fences"},
    {"cell_number": 5, "submitted_type": "code", "required_type": "code",
     "defect": "defines singular predict_label; capitalizes labels and can change the vocabulary; writes final.csv with the wrong schema and default index; omits row_id, exact filename, contract checks, and stage trace",
     "repair": "define exact predict_labels returning an index-preserving Series of original labels; verify the contract; write p18_predictions.csv with row_id and prediction, index=False; append refit and package stages"},
    {"cell_number": 6, "submitted_type": "text", "required_type": "text",
     "defect": "writeup omits the pinned protocol, true metric, full refit, measured alternative, and limitation; asserts that no alternative matters; illegally enables a Round 1 GPU",
     "repair": "report the corrected bounded recipe and macro-F1, one measured alternative, full refit, and validation-selection limitation; state Round 1 CPU only and Round 2 Colab L4/GPU permitted"},
]
cell_plan = [
    ("text", "corrected Approach, Intuition, Alternatives, and limitation"),
    ("code", "imports, data, pinned carve, raw audit, and bounded macro-F1 selection"),
    ("code", "full-data refit and executable predict_labels contract"),
    ("text", "matched fenced predict_labels excerpt"),
    ("text", "display macro-F1 formula and equal-class interpretation"),
    ("code", "write and verify exact audit and prediction CSV artifacts"),
    ("text", "metric-audit sentence and Round 1/Round 2 policy"),
]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)

raw_trials = []
for k in range(1, 52, 2):
    trial = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    trial.fit(X_train, y_train)
    raw_trials.append((trial.score(X_val, y_val), k, trial))
claimed_score, raw_selected_k, raw_selected_model = max(raw_trials)
claimed_score = float(raw_selected_model.score(X_val, y_val))
raw_selected_macro_f1 = float(f1_score(
    y_val, raw_selected_model.predict(X_val), average="macro"
))

selection_rows = []
for k in K_GRID:
    candidate = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ])
    candidate.fit(X_train, y_train)
    value = float(f1_score(y_val, candidate.predict(X_val), average="macro"))
    selection_rows.append({"k": int(k), "val_f1": value, "selected": False})
best_row = max(selection_rows, key=lambda row: (row["val_f1"], -row["k"]))
selected_k = int(best_row["k"])
corrected_val_f1 = float(best_row["val_f1"])
best_row["selected"] = True
selection_log = pd.DataFrame(selection_rows, columns=["k", "val_f1", "selected"])
stage_trace.append("select")

In [ ]:
final_model = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=selected_k)),
])
final_model.fit(X, y)

def predict_labels(X_test):
    predictions = final_model.predict(X_test)
    return pd.Series(predictions, index=X_test.index)

stage_trace.append("refit")
probe = X.iloc[50:90]
probe_predictions = predict_labels(probe)
contract_checks = {
    "series": isinstance(probe_predictions, pd.Series),
    "length": len(probe_predictions) == len(probe),
    "index": probe_predictions.index.equals(probe.index),
    "vocab": set(probe_predictions.unique()) <= set(np.unique(y)),
}
assert all(contract_checks.values())
contract_ok = all(contract_checks.values())

In [ ]:
pd.DataFrame(cell_audit, columns=[
    "cell_number", "submitted_type", "required_type", "defect", "repair"
]).to_csv("p18_audit.csv", index=False)
pd.DataFrame({"row_id": probe.index,
              "prediction": probe_predictions.to_numpy()}).to_csv(
    "p18_predictions.csv", index=False
)
stage_trace.append("package")

saved_audit = pd.read_csv("p18_audit.csv")
saved_predictions = pd.read_csv("p18_predictions.csv")
assert list(saved_audit.columns) == ["cell_number", "submitted_type", "required_type", "defect", "repair"]
assert saved_audit["cell_number"].tolist() == [1, 2, 3, 4, 5, 6]
assert saved_audit["submitted_type"].tolist() == ["code", "text", "code", "text", "code", "text"]
assert set(saved_audit["required_type"]) <= {"text", "code", "split"}
assert saved_audit["defect"].str.strip().ne("").all()
assert saved_audit["repair"].str.strip().ne("").all()
assert list(saved_predictions.columns) == ["row_id", "prediction"]
assert len(saved_predictions) == 40
assert saved_predictions["row_id"].tolist() == probe.index.tolist()
assert saved_predictions["prediction"].tolist() == probe_predictions.tolist()
assert stage_trace == ["setup", "select", "refit", "package"]
assert np.isclose(claimed_score, 0.8266666666666667, atol=1e-12, rtol=0)
assert np.isclose(raw_selected_macro_f1, 0.8103481812876873, atol=1e-12, rtol=0)
assert np.isclose(corrected_val_f1, 0.8103481812876873, atol=1e-12, rtol=0)

## Row 1 — Corrected writeup

### Approach

Use one pinned stratified validation carve (`test_size=150`, `random_state=20260804`) and evaluate exactly scaled kNN candidates `k ∈ {5, 7, 9, 11, 15}` by validation macro-F1, selecting the smallest `k` on ties. The corrected recipe selects scaled 11-NN with `corrected_val_f1 = 0.8103481812876873`, then refits that pipeline on all 600 labeled rows before exposing `predict_labels`.

### Intuition

Scaling makes each feature's units comparable in the distance calculation, and the bounded grid controls how local or smooth the neighbor vote is. Macro-F1 gives both classes equal weight, which is important for the 2:1 class balance.

### Alternatives

A measured alternative, scaled 5-NN, scored `0.7665823769694612`; scaled 7-NN scored `0.7939560439560439`. Because the grid winner was chosen on this same validation carve, `0.8103481812876873` may be optimistic and is not an untouched-test estimate.

## Row 3 — Repaired fenced excerpt

```python
def predict_labels(X_test):
    predictions = final_model.predict(X_test)
    return pd.Series(predictions, index=X_test.index)
```

## Row 4 — Repaired formula and interpretation

Let $K$ be the number of classes and let $F_{1,k}$ be the precision–recall harmonic mean computed with class $k$ treated as the target. Then

$$
F_{1,\mathrm{macro}} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}.
$$

Every class contributes one term with the same weight, so the larger class in the 2:1 task cannot dominate the metric merely by having more examples.

## Metric-audit sentence

Neutral `claimed_score` is classifier accuracy: for the raw draft's selected 11-NN model it is `0.8266666666666667`, so calling it macro-F1 was wrong; that same raw-selected model has `raw_selected_macro_f1 = 0.8103481812876873`. Separately, the corrected bounded-grid recipe selects 11-NN with `corrected_val_f1 = 0.8103481812876873`.

## Row 6 — Round-policy diagnosis

Setting or claiming `RUNTIME = "GPU"` for Round 1 is illegal even if Colab exposes that hardware. **Round 1 is CPU only; Round 2 permits Colab L4/GPU.**

### Answer check

The six-row audit, raw accuracy-versus-macro-F1 reproduction, five-row corrected selection log, full-data refit, exact prediction contract, two CSV artifacts, and four-stage trace are checked; floating comparisons use `atol=1e-12` and `rtol=0`.